In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from dateutil.parser import parser
from sklearn.preprocessing import MinMaxScaler


In [ ]:
# Get all the data
df_banks = pd.read_csv("data/banklist.csv")
df_banks_old = pd.read_csv("data/bank_failues.csv")
df_recc = pd.read_csv("data/USREC.csv")
df_banks


In [ ]:
df_banks_old


In [ ]:
# Combine the old data with the newer data to make a singular dataframe
psr = parser()

df1 = df_banks.rename(columns={"Closing Date": "Date", "Bank Name": "Name"})
df2 = df_banks_old.rename(columns={"Failure Date": "Date", "Institution Name": "Name"})

df1["Date"] = df1["Date"].apply(lambda x: psr.parse(x))
df2["Date"] = df2["Date"].apply(lambda x: psr.parse(x))

df = pd.concat([df1, df2])
df.set_index("Date", inplace=True)
df.sort_index(axis=0, inplace=True)
df


In [ ]:
# combine the recession dates with the bank failure data
df_recc.rename(columns={"USREC": "Recession", "DATE": "Date"}, inplace=True)
df_recc["Date"] = pd.to_datetime(df_recc["Date"])
df_recc = df_recc.set_index("Date")
df_rec_res = df_recc.resample("M").sum()

# combine it!
df["Recession"] = 0

rec_dates = df_rec_res[df_rec_res["Recession"] == 1].index
# convert all dates to use month

rec_dates = [
    pd.Timestamp(year=_date.year, month=_date.month, day=1) for _date in rec_dates
]

for i, _date in enumerate(df.index):
    if _date in rec_dates:
        df["Recession"].loc[i] = 1

print(df[df["Recession"] == 1])

# result = df.join(df_rec_res, on=["Date", "Recession"], how="right")
# result
# for d in df['Date']:
# if the rec month is the same month in df['Date']
# set the 'Recession' column to 1, else 0


In [ ]:
df["Count"] = 1
df = df.resample("M").sum()
df = df[["Count"]]
scaler = MinMaxScaler()
df["Bank Failures"] = scaler.fit_transform(df[["Count"]])

df_data = df.drop("Count", axis=1)

df_recc = df_recc[df_recc.index >= df_data.index[0]]

# plot the data
fig, ax = plt.subplots(figsize=(15, 5))
df_recc.plot.area(ax=ax, alpha=0.5, color="gray")
df_data.plot(ax=ax, title="Bank Failures Over Time")
plt.show()


In [ ]:
# look at bank failures in the last 4 years
fig, ax = plt.subplots(figsize=(15, 5))
scaler = MinMaxScaler()

df1 = df_recc["Recession"].iloc[-4 * 12 :]
df2 = df["Bank Failures"].iloc[-4 * 12 :]

# count the number of bank failures in this new time frame


df2["Bank Failures"] = scaler.fit_transform(df2[["Count"]])
df2 = df2.drop("Count", axis=1)

df1.plot.area(ax=ax, alpha=0.5, color="gray")
df2.plot(ax=ax)
plt.show()
